In [1]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,SimpleRNN

In [2]:
model = Sequential()
model.add(SimpleRNN(3,input_shape = (4,5)))
model.add(Dense(1,activation = 'sigmoid'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 3)              │            27 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31 (124.00 B)

 Trainable params: 31 (124.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
print(model.get_weights()[0].shape)
model.get_weights()[0]

(5, 3)


array([[ 0.14903957, -0.5573329 , -0.07104981],
       [-0.75375915, -0.60820204,  0.852459  ],
       [-0.2105751 ,  0.7670259 ,  0.75018233],
       [-0.80700386,  0.81091684,  0.6237753 ],
       [ 0.03022528, -0.78447396,  0.6049517 ]], dtype=float32)

In [7]:
print(model.get_weights()[1].shape)
model.get_weights()[1]

(3, 3)


array([[-0.9628155 ,  0.05427638,  0.26465094],
       [ 0.22716583, -0.36756748,  0.90182596],
       [-0.1462249 , -0.9284118 , -0.34156984]], dtype=float32)

In [8]:
print(model.get_weights()[2].shape)
model.get_weights()[2]

(3,)


array([0., 0., 0.], dtype=float32)

In [9]:
print(model.get_weights()[3].shape)
model.get_weights()[3]

(3, 1)


array([[-0.49471498],
       [ 0.9219855 ],
       [ 0.41897082]], dtype=float32)

In [11]:
print(model.get_weights()[4].shape)
model.get_weights()[4]

(1,)


array([0.], dtype=float32)

In [ ]:
import numpy as np

# Extract learned/initialized weights from Keras model
W_xh = model.get_weights()[0]  # shape: (5, 3) - input kernel
W_hh = model.get_weights()[1]  # shape: (3, 3) - recurrent kernel
b_h  = model.get_weights()[2]  # shape: (3,)   - hidden bias
W_hy = model.get_weights()[3]  # shape: (3, 1) - dense kernel
b_y  = model.get_weights()[4]  # shape: (1,)   - dense bias

# Sample input sequence: batch_size=1, timesteps=4, features=5
np.random.seed(42)
X_sample = np.random.randn(1, 4, 5)

# 1. Keras prediction
keras_output = model.predict(X_sample)
print("Keras SimpleRNN Output:", keras_output)

# 2. Step-by-step manual forward pass
# Initial hidden state h_0 is all zeros
h_t = np.zeros((1, 3))

for t in range(4):
    x_t = X_sample[:, t, :]  # shape: (1, 5)
    # Recurrence equation: h_t = tanh(x_t @ W_xh + h_{t-1} @ W_hh + b_h)
    h_t = np.tanh(np.dot(x_t, W_xh) + np.dot(h_t, W_hh) + b_h)
    print(f"Timestep {t+1} hidden state:", h_t)

# Output layer: sigmoid(h_4 @ W_hy + b_y)
z = np.dot(h_t, W_hy) + b_y
manual_output = 1.0 / (1.0 + np.exp(-z))
print("\nManual Forward Pass Output:", manual_output)

# Mathematical verification check
diff = np.abs(keras_output - manual_output).max()
print(f"Max absolute difference: {diff:.8e}")
print("Verification:", "PASSED" if diff < 1e-6 else "FAILED")

### Recurrence Equations & Weight Breakdown
- **Input-to-Hidden**: $W_{xh} \in \mathbb{R}^{5 \times 3}$
- **Hidden-to-Hidden**: $W_{hh} \in \mathbb{R}^{3 \times 3}$
- **Hidden Bias**: $b_h \in \mathbb{R}^{3}$
- **State Transition**: $h_t = \tanh(x_t W_{xh} + h_{t-1} W_{hh} + b_h)$
- **Dense Output**: $\hat{y} = \sigma(h_T W_{hy} + b_y)$

The manual step-by-step computation exactly mirrors Keras SimpleRNN, confirming how parameter sharing operates identically across all timesteps.